## 문제 1 · CSV 읽기와 데이터 확인

In [ ]:
import math
import pandas as pd

def read_house_csv(filename):
    print(f"실행 파일명: {filename}")
    try:
        data = pd.read_csv(filename)
    except FileNotFoundError:
        print("파일명과 업로드 여부를 확인하세요.")
        return None
    except (pd.errors.EmptyDataError, pd.errors.ParserError, UnicodeError) as error:
        print(f"CSV 읽기 오류: {error}")
        return None

    if "price" not in data.columns:
        print("필수 열 누락: price")
        return None

    if len(data) == 0:
        print("전체 데이터 행 수: 0")
        return data

    print("첫 번째 행")
    print(data.iloc[[0]].to_string(index=False))
    print(f"전체 데이터 행 수: {len(data)}")
    print("첫 번째 행 price의 자료형:", type(data["price"].iloc[0]))
    return data

df_house = read_house_csv("1.01.house.price.csv")
print()
# 오류 테스트의 결과가 정상 데이터프레임을 덮어쓰지 않도록 별도로 호출합니다.
read_house_csv("없는_주택_파일.csv");

실행 파일명: 1.01.house.price.csv
첫 번째 행
 ID  size  year  price
  1    34     5    5.0
전체 데이터 행 수: 10
첫 번째 행 price의 자료형: <class 'numpy.float64'>

실행 파일명: 없는_주택_파일.csv
파일명과 업로드 여부를 확인하세요.


## 문제 2 · 행별 자료형 변환

In [ ]:
houses = []
excluded_count = 0

if df_house is None:
    print("CSV를 읽은 뒤 다시 실행하세요.")
else:
    for row_number, (_, row) in enumerate(df_house.iterrows(), start=1):
        house = {}
        try:
            for column, convert in (("ID", int), ("size", int), ("year", int), ("price", float)):
                if column not in row.index or pd.isna(row[column]):
                    raise ValueError("필수 값 누락")

                value = row[column]
                if not math.isfinite(float(value)):
                    raise ValueError("유한한 숫자가 아님")

                # int()가 소수 부분을 조용히 잘라내는 경우를 방지합니다.
                if convert is int:
                    if isinstance(value, str):
                        value = int(value)
                    elif value != int(value):
                        raise ValueError("정수가 아님")
                house[column] = convert(value)
        except (ValueError, TypeError, OverflowError) as error:
            excluded_count += 1
            print(f"데이터 {row_number}행 제외 ({column}): {error}")
            continue

        houses.append(house)

    print(f"정상 저장 행 수: {len(houses)}")
    print(f"제외 행 수: {excluded_count}")
    if houses:
        print("첫 번째 저장 행:", houses[0])
        print("자료형:", type(houses[0]["ID"]), type(houses[0]["size"]),
              type(houses[0]["year"]), type(houses[0]["price"]))

정상 저장 행 수: 10
제외 행 수: 0
첫 번째 저장 행: {'ID': 1, 'size': 34, 'year': 5, 'price': 5.0}
자료형: <class 'int'> <class 'int'> <class 'int'> <class 'float'>


## 문제 3 · 유효한 주택 필터링

In [ ]:
houses_test = houses.copy()
houses_test.extend([
    {"ID": 101, "year": 5, "price": 3},
    {"ID": 102, "size": 35, "year": 5, "price": None},
    {"ID": 103, "size": "오류", "year": 5, "price": 3},
    {"ID": 104, "size": 35, "year": 5, "price": 3},
])
selected = []

for house in houses_test:
    house_id = house.get("ID", "미상")
    if "size" not in house or "price" not in house:
        missing_key = "size" if "size" not in house else "price"
        print(f"ID {house_id} 제외: {missing_key} 키 누락")
        continue

    if pd.isna(house["size"]) or pd.isna(house["price"]):
        missing_value = "size" if pd.isna(house["size"]) else "price"
        print(f"ID {house_id} 제외: {missing_value} 값 누락")
        continue

    try:
        size = float(house["size"])
        price = float(house["price"])
    except (ValueError, TypeError, OverflowError):
        print(f"ID {house_id} 제외: size 또는 price를 숫자로 변환할 수 없음")
        continue

    if not math.isfinite(size) or not math.isfinite(price):
        print(f"ID {house_id} 제외: size 또는 price가 유한한 숫자가 아님")
        continue

    if size <= 0 or price < 0:
        print(f"ID {house_id} 제외: size가 0 이하이거나 price가 음수")
        continue

    if size >= 30 and price <= 4:
        selected.append(house)

for house in selected:
    print(f"선택 ID: {house['ID']}")
print(f"전체 선택 건수: {len(selected)}")

ID 101 제외: size 키 누락
ID 102 제외: price 값 누락
ID 103 제외: size 또는 price를 숫자로 변환할 수 없음
선택 ID: 3
선택 ID: 4
선택 ID: 5
선택 ID: 9
선택 ID: 10
선택 ID: 104
전체 선택 건수: 6


## 문제 4 · 크기당 가격 함수와 예외 처리

In [ ]:
def unit_price(price, size):
    try:
        price = float(price)
        size = float(size)
    except ValueError:
        print("값 오류: 숫자로 변환할 수 없는 문자열입니다.")
        return None
    except TypeError:
        print("자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.")
        return None
    except OverflowError:
        print("값 오류: 표현할 수 있는 숫자 범위를 벗어났습니다.")
        return None

    if not math.isfinite(price) or not math.isfinite(size):
        print("값 오류: 유한한 숫자를 입력하세요.")
        return None
    if size <= 0:
        print("크기 오류: 크기는 0보다 커야 합니다.")
        return None
    if price < 0:
        print("가격 오류: 가격은 0 이상이어야 합니다.")
        return None

    return price / size

print("전체 주택의 크기당 가격")
for house in houses:
    result = unit_price(house["price"], house["size"])
    print(f"ID {house['ID']} → {result}")

print()
print("테스트 호출 7개")
test_inputs = [(4, 40), ("4", "40"), (4, 0), (-4, 40),
               ("오류", 40), (None, 40), (6, 30)]
for price, size in test_inputs:
    print(f"unit_price({price!r}, {size!r})")
    print("반환값:", unit_price(price, size))

전체 주택의 크기당 가격
ID 1 → 0.14705882352941177
ID 2 → 0.1
ID 3 → 0.13333333333333333
ID 4 → 0.07894736842105263
ID 5 → 0.075
ID 6 → 0.08750000000000001
ID 7 → 0.08846153846153845
ID 8 → 0.1
ID 9 → 0.08823529411764706
ID 10 → 0.08529411764705883

테스트 호출 7개
unit_price(4, 40)
반환값: 0.1
unit_price('4', '40')
반환값: 0.1
unit_price(4, 0)
크기 오류: 크기는 0보다 커야 합니다.
반환값: None
unit_price(-4, 40)
가격 오류: 가격은 0 이상이어야 합니다.
반환값: None
unit_price('오류', 40)
값 오류: 숫자로 변환할 수 없는 문자열입니다.
반환값: None
unit_price(None, 40)
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
반환값: None
unit_price(6, 30)
반환값: 0.2


## 문제 5 · 목적에 맞는 자료구조

In [ ]:
# 리스트는 입력 순서를 보존하고 같은 ID가 있어도 모두 기록합니다.
# 집합은 중복을 제거하고 입력 순서대로 조회하는 용도로 쓰기 어렵습니다.
house_ids = []

# 집합은 같은 year를 여러 번 추가해도 하나만 저장합니다.
# 리스트는 중복을 허용하므로 중복 여부를 직접 검사해야 합니다.
unique_years = set()

# 딕셔너리는 ID를 키로 사용해 가격을 바로 조회합니다.
# 리스트는 ID와 인덱스가 다를 수 있어 해당 ID를 찾으려면 행을 탐색해야 합니다.
prices_by_id = {}

for house in houses:
    house_ids.append(house["ID"])
    unique_years.add(house["year"])
    prices_by_id[house["ID"]] = house["price"]

# 튜플은 원소를 추가, 삭제, 교체할 수 없어 고정된 두 조건을 저장하기 적합합니다.
# 리스트는 원소를 추가, 삭제, 교체할 수 있습니다.
search_conditions = (30, 4)

print("전체 주택 ID:", house_ids)
print("중복 없는 year:", unique_years)
if 3 in prices_by_id:
    print("ID 3의 가격:", prices_by_id[3])
else:
    print("ID 3의 가격: 해당 ID가 없습니다.")
print("검색 조건 (최소 크기, 최대 가격):", search_conditions)

전체 주택 ID: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
중복 없는 year: {2, 5, 12, 18, 19, 20, 22}
ID 3의 가격: 4.0
검색 조건 (최소 크기, 최대 가격): (30, 4)
